In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/name-that-llm/sample_submission.parquet
/kaggle/input/competitions/name-that-llm/train.csv
/kaggle/input/competitions/name-that-llm/test.csv


In [2]:
import pandas as pd, numpy as np, re
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

D = '/kaggle/input/competitions/name-that-llm/'
train = pd.read_csv(D + 'train.csv')
test  = pd.read_csv(D + 'test.csv')
print(train.shape, test.shape)
print(train.model.value_counts())

(2699, 4) (3, 2)
model
qwen3.8-max               150
claude-sonnet-5           150
gemini-3.6-flash          150
gpt-5.6-terra             150
glm-5.3-flash             150
gemini-3.1-pro-preview    150
gemini-3.8-flash          150
claude-opus-4-6           150
gpt-5.6-luna              150
claude-opus-4-8           150
gemini-3.7-flash          150
claude-opus-5             150
kimi-k3                   150
gpt-5.5                   150
glm-5.3                   150
gpt-6-astra               150
gpt-5.6-sol               150
claude-fable-5            149
Name: count, dtype: int64


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler, FunctionTransformer
import scipy.sparse as sp

def style_feats(texts):
    rows = []
    for t in texts:
        words = t.split()
        n_w = max(len(words), 1)
        sents = [s for s in re.split(r'[.!?]+', t) if s.strip()]
        rows.append([
            t.count('—') / n_w, t.count('–') / n_w, t.count(',') / n_w,
            t.count(';') / n_w, t.count(':') / n_w, t.count('(') / n_w,
            t.count('"') / n_w, t.count('“') / n_w, t.count("’") / n_w,
            t.count("'") / n_w, t.count('\n') / n_w,
            np.mean([len(w) for w in words]) if words else 0,
            n_w / max(len(sents), 1),
            sum(c.isupper() for c in t) / max(len(t), 1),
            sum(c.isdigit() for c in t) / max(len(t), 1),
            sum(ord(c) > 127 for c in t) / max(len(t), 1),
        ])
    return np.array(rows)

def make_word():
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                        lowercase=False, token_pattern=r"(?u)\b\w+\b|[^\w\s]"),
        LogisticRegression(C=30, max_iter=3000, class_weight='balanced'))

def make_char():
    return make_pipeline(
        TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=3,
                        sublinear_tf=True, lowercase=False, max_features=300_000),
        LogisticRegression(C=30, max_iter=3000, class_weight='balanced'))

def make_style():
    return make_pipeline(
        FunctionTransformer(style_feats), StandardScaler(),
        LogisticRegression(C=1, max_iter=3000, class_weight='balanced'))

makers = {'word': make_word, 'char': make_char, 'style': make_style}

In [5]:
X = train.text.values
classes = np.array(sorted(train.model.unique()))
y = pd.Categorical(train.model, categories=classes).codes
groups = train.prompt_id.values

gkf = GroupKFold(n_splits=5)
oof = {k: np.zeros((len(train), len(classes))) for k in makers}

for fold, (tr, va) in enumerate(gkf.split(X, y, groups)):
    for k, mk in makers.items():
        m = mk().fit(X[tr], y[tr])
        p = np.zeros((len(va), len(classes)))
        p[:, m.classes_] = m.predict_proba(X[va])
        oof[k][va] = p
    print('fold', fold, 'done')

for k in oof:
    print(k, balanced_accuracy_score(y, oof[k].argmax(1)))

# find blend weights (simple grid)
best = (0, None)
for w1 in np.arange(0, 1.01, 0.1):
    for w2 in np.arange(0, 1.01 - w1, 0.1):
        w3 = 1 - w1 - w2
        p = w1 * oof['word'] + w2 * oof['char'] + w3 * oof['style']
        s = balanced_accuracy_score(y, p.argmax(1))
        if s > best[0]: best = (s, (w1, w2, w3))
print('best blend', best)

fold 0 done
fold 1 done
fold 2 done
fold 3 done
fold 4 done
word 0.33300521998508575
char 0.32745960725826495
style 0.261866766094954
best blend (np.float64(0.3730027342779021), (np.float64(0.4), np.float64(0.30000000000000004), np.float64(0.29999999999999993)))


In [6]:
w = best[1]
P = np.zeros((len(test), len(classes)))
for wk, (k, mk) in zip(w, makers.items()):
    m = mk().fit(X, y)
    p = np.zeros((len(test), len(classes)))
    p[:, m.classes_] = m.predict_proba(test.text.values)
    P += wk * p

submission = pd.DataFrame({'id': test['id'], 'model': classes[P.argmax(1)]})
submission.to_parquet('submission.parquet', index=False)
print(submission.head(), submission.model.nunique())

             id             model
0  00353fb1c541     gpt-5.6-terra
1  005f5a9a6139       gpt-6-astra
2  0062a53d5487  gemini-3.7-flash 3
